# M5 Demand Forecasting → Inventory Decisions
## Notebook 01 — Data Preparation

**Goal of this notebook:** Load the raw M5 (Walmart) data, keep only the subset we're modelling (Texas stores, FOODS category), reshape it into one row per item-store-day, attach calendar and price information, and save a clean, memory-efficient file for the rest of the project.

**Subset:** 3 Texas stores (TX_1, TX_2, TX_3) × FOODS items × ~1,941 days

**Output:** `m5_tx_foods_long.parquet`

| Step | What happens |
|---|---|
| 0 | Set up environment and download data |
| 1 | Inspect the raw files |
| 2 | Filter to Texas + FOODS |
| 3 | Reshape wide → long |
| 4 | Merge calendar (dates, events, SNAP) |
| 5 | Merge sell prices |
| 6 | Reduce memory and save |

### Business Context

A store manager must decide how much of each product to stock. Ordering too little causes **stockouts** (lost sales, unhappy customers); ordering too much causes **holding costs** (storage, tied-up cash, spoilage — especially for food).

This project turns demand forecasts into ordering decisions in three layers:
1. **Point forecast** — expected daily unit sales per item-store, 28 days ahead
2. **Uncertainty** — quantile forecasts (P50 / P90) to capture how high demand could go
3. **Inventory policy** — safety stock and reorder points, backtested against a naive policy to compare stockouts vs. holding cost

**Dataset:** [M5 Forecasting (Walmart)](https://www.kaggle.com/competitions/m5-forecasting-accuracy) — daily unit sales for 3,049 products across 10 stores in CA, TX, WI (Jan 2011 – May 2016).

## Step 0 — Environment Setup & Data Download

The M5 data is hosted as a Kaggle competition. We authenticate using a Kaggle API token stored in Colab **Secrets** (never hard-coded), then download the competition files.

> ⚠️ Run the install cell, then **Runtime → Restart session**, then continue from the next cell.

In [1]:
# Upgrade kagglehub — new-style Kaggle API tokens need version >= 0.4.1
!pip install -q --upgrade kagglehub

In [2]:
import os
from google.colab import userdata
import kagglehub

# Read the Kaggle token from Colab Secrets
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')

# Download M5 competition files; returns the local folder path
DATA_PATH = kagglehub.competition_download('m5-forecasting-accuracy')
print("Data folder:", DATA_PATH, "\n")

# List each file with its size so we know what we're dealing with
for f in sorted(os.listdir(DATA_PATH)):
    size_mb = os.path.getsize(os.path.join(DATA_PATH, f)) / 1e6
    print(f"{f:35s} {size_mb:8.1f} MB")

Data folder: /root/.cache/kagglehub/competitions/m5-forecasting-accuracy 

calendar.csv                             0.1 MB
sales_train_evaluation.csv             121.7 MB
sales_train_validation.csv             120.0 MB
sample_submission.csv                    5.2 MB
sell_prices.csv                        203.4 MB


## Step 1 — Inspect the Raw Files

Before transforming anything, we load each file and check its shape, memory footprint, and structure.

### Data Dictionary

**`sales_train_evaluation.csv`** — one row per item-store (30,490 series), one column per day
| Column | Meaning |
|---|---|
| `id` | Unique series ID, e.g. `FOODS_3_090_TX_1_evaluation` |
| `item_id` | Product, e.g. `FOODS_3_090` |
| `dept_id` | Department, e.g. `FOODS_3` |
| `cat_id` | Category: FOODS, HOBBIES, HOUSEHOLD |
| `store_id` | Store, e.g. `TX_1` |
| `state_id` | State: CA, TX, WI |
| `d_1` … `d_1941` | Units sold on day 1 … day 1941 |

**`calendar.csv`** — one row per day (1,969 days; the last 28 have no sales — they were the competition's forecast period)
| Column | Meaning |
|---|---|
| `date` | Calendar date |
| `d` | Day label (`d_1`, `d_2`, …) — links to sales columns |
| `wm_yr_wk` | Walmart week ID — links to prices |
| `weekday`, `wday`, `month`, `year` | Date parts |
| `event_name_1/2`, `event_type_1/2` | Holidays & events (e.g. SuperBowl, Easter) |
| `snap_CA`, `snap_TX`, `snap_WI` | 1 if SNAP food-assistance benefits were paid that day in that state |

**`sell_prices.csv`** — one row per store-item-week
| Column | Meaning |
|---|---|
| `store_id`, `item_id` | Which product in which store |
| `wm_yr_wk` | Walmart week ID — links to calendar |
| `sell_price` | Price that week (missing weeks = item not on sale yet) |

In [3]:
import pandas as pd
import numpy as np

# Load the three files we need
calendar = pd.read_csv(os.path.join(DATA_PATH, 'calendar.csv'))
sales    = pd.read_csv(os.path.join(DATA_PATH, 'sales_train_evaluation.csv'))
prices   = pd.read_csv(os.path.join(DATA_PATH, 'sell_prices.csv'))

def describe_df(name, df):
    """Print rows, columns, and memory usage of a DataFrame."""
    mem_mb = df.memory_usage(deep=True).sum() / 1e6
    print(f"{name:10s} rows: {df.shape[0]:>10,}   cols: {df.shape[1]:>5,}   memory: {mem_mb:8.1f} MB")

describe_df('calendar', calendar)
describe_df('sales',    sales)
describe_df('prices',   prices)

calendar   rows:      1,969   cols:    14   memory:      0.7 MB
sales      rows:     30,490   cols: 1,947   memory:    484.3 MB
prices     rows:  6,841,121   cols:     4   memory:    894.6 MB


In [4]:
print("Unique items:", sales['item_id'].nunique())
print("Stores:      ", sorted(sales['store_id'].unique()))
print("Categories:  ", sorted(sales['cat_id'].unique()))
print("Departments: ", sorted(sales['dept_id'].unique()))

print("\nNumber of series (item-store rows) per category:")
print(sales['cat_id'].value_counts())

Unique items: 3049
Stores:       ['CA_1', 'CA_2', 'CA_3', 'CA_4', 'TX_1', 'TX_2', 'TX_3', 'WI_1', 'WI_2', 'WI_3']
Categories:   ['FOODS', 'HOBBIES', 'HOUSEHOLD']
Departments:  ['FOODS_1', 'FOODS_2', 'FOODS_3', 'HOBBIES_1', 'HOBBIES_2', 'HOUSEHOLD_1', 'HOUSEHOLD_2']

Number of series (item-store rows) per category:
cat_id
FOODS        14370
HOUSEHOLD    10470
HOBBIES       5650
Name: count, dtype: int64


In [5]:
# Show the 6 ID columns, the first 4 days, and the last 3 days
sales.iloc[:5, list(range(10)) + [-3, -2, -1]]

,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,d_1939,d_1940,d_1941
0,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,3,0,1
1,HOBBIES_1_002_CA_1_evaluation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,0,0,0
2,HOBBIES_1_003_CA_1_evaluation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,3,0,1
3,HOBBIES_1_004_CA_1_evaluation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,0,2,6
4,HOBBIES_1_005_CA_1_evaluation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,2,1,0


In [6]:
calendar.head()

,date,wm_yr_wk,weekday,wday,month,year,d,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
0,2011-01-29,11101,Saturday,1,1,2011,d_1,NaN,NaN,NaN,NaN,0,0,0
1,2011-01-30,11101,Sunday,2,1,2011,d_2,NaN,NaN,NaN,NaN,0,0,0
2,2011-01-31,11101,Monday,3,1,2011,d_3,NaN,NaN,NaN,NaN,0,0,0
3,2011-02-01,11101,Tuesday,4,2,2011,d_4,NaN,NaN,NaN,NaN,1,1,0
4,2011-02-02,11101,Wednesday,5,2,2011,d_5,NaN,NaN,NaN,NaN,1,0,1


In [7]:
prices.head()

,store_id,item_id,wm_yr_wk,sell_price
0,CA_1,HOBBIES_1_001,11325,9.58
1,CA_1,HOBBIES_1_001,11326,9.58
2,CA_1,HOBBIES_1_001,11327,8.26
3,CA_1,HOBBIES_1_001,11328,8.26
4,CA_1,HOBBIES_1_001,11329,8.26


 ### Step 1 — Key Observations

- **Memory is the main constraint.** `prices` uses ~895 MB despite having only 4 columns, because ID columns are stored as Python strings. `sales` uses ~484 MB because every daily count is stored as a 64-bit integer.
- **FOODS is the largest category** (14,370 of 30,490 series = 1,437 items × 10 stores). Our Texas subset will be 1,437 items × 3 stores = **4,311 series**.
- **Not every zero is real zero demand.** Prices begin when an item is first stocked (e.g. `HOBBIES_1_001` in `CA_1` has no price until week 11325, mid-2013). Zero sales *before* an item's first price mean "not on the shelf", not "no demand" — these must be handled separately.
- **Walmart weeks start on Saturday** (`wday = 1` is Saturday).

## Step 2 — Filter to Texas + FOODS

We keep only the 3 Texas stores and the FOODS category, then drop the full tables from memory. Filtering **before** reshaping is essential: reshaping all 30,490 series to daily rows would produce ~59M rows, versus ~8.4M for our subset.

In [8]:
STATE = 'TX'
CATEGORY = 'FOODS'

# Keep only rows where BOTH conditions are true
sales_tx = sales[(sales['state_id'] == STATE) & (sales['cat_id'] == CATEGORY)].reset_index(drop=True)

print(f"Kept {len(sales_tx):,} of {len(sales):,} series")

Kept 4,311 of 30,490 series


In [9]:
# The exact items and stores that survived the sales filter
tx_stores = sales_tx['store_id'].unique()
tx_food_items = sales_tx['item_id'].unique()

prices_tx = prices[
    prices['store_id'].isin(tx_stores) &
    prices['item_id'].isin(tx_food_items)
].reset_index(drop=True)

print(f"Kept {len(prices_tx):,} of {len(prices):,} price rows")

Kept 987,960 of 6,841,121 price rows


In [10]:
import gc

# Delete the full tables, we no longer need them
del sales, prices
gc.collect()

describe_df('sales_tx', sales_tx)
describe_df('prices_tx', prices_tx)

# Sanity checks: the code stops with an error if any of these are false
assert set(sales_tx['store_id'].unique()) == {'TX_1', 'TX_2', 'TX_3'}
assert set(sales_tx['cat_id'].unique()) == {'FOODS'}
assert sales_tx['item_id'].nunique() == 1437
assert len(sales_tx) == 1437 * 3

print("\n✅ All sanity checks passed")

sales_tx   rows:      4,311   cols: 1,947   memory:     68.5 MB
prices_tx  rows:    987,960   cols:     4   memory:    127.4 MB

✅ All sanity checks passed


### Step 2 — Result

Kept **4,311 series** (1,437 items × 3 stores) and **987,960 price rows**. Memory dropped from **~1,380 MB → ~196 MB** (−86%) just by filtering.

---

## Step 3 — Reshape Wide → Long

Models learn from rows, so we convert from one-row-per-series (1,941 day columns) to **one row per item-store-day**. This expands the data to 4,311 × 1,941 = **8,367,651 rows**.

To keep memory under control, we prepare the wide table *before* melting:
- Drop columns with no information (`id`, `cat_id`, `state_id`)
- Store ID columns as `category` instead of text
- Store daily sales as `int16` instead of `int64`
- Rename day columns `d_1 … d_1941` → integers `1 … 1941`

In [11]:
# All the day columns: 'd_1', 'd_2', ..., 'd_1941'
day_cols = [c for c in sales_tx.columns if c.startswith('d_')]

# Remember total units sold so we can verify nothing gets lost in the reshape
total_units_before = sales_tx[day_cols].sum().sum()
max_daily_sales = sales_tx[day_cols].max().max()
print(f"Total units sold: {total_units_before:,}")
print(f"Max units sold by one item in one store on one day: {max_daily_sales}")

# 1) Drop columns that carry no information
sales_tx = sales_tx.drop(columns=['id', 'cat_id', 'state_id'])

# 2) Text ID columns -> category
for col in ['item_id', 'dept_id', 'store_id']:
    sales_tx[col] = sales_tx[col].astype('category')

# 3) Daily sales -> int16 (safe because max is far below 32,767)
assert max_daily_sales < 32_767
sales_tx[day_cols] = sales_tx[day_cols].astype('int16')

# 4) Rename 'd_1' -> 1, 'd_2' -> 2, ...
sales_tx = sales_tx.rename(columns={c: int(c[2:]) for c in day_cols})

describe_df('sales_tx', sales_tx)

Total units sold: 13,172,106
Max units sold by one item in one store on one day: 634
sales_tx   rows:      4,311   cols: 1,944   memory:     16.9 MB


In [12]:
df = sales_tx.melt(
    id_vars=['item_id', 'dept_id', 'store_id'],   # columns to keep as-is (repeated on every row)
    var_name='d',                                 # name for the new "which day" column
    value_name='sales'                            # name for the new "units sold" column
)

# The day column comes out as a generic type; make it a compact integer
df['d'] = df['d'].astype('int16')

# We no longer need the wide version
del sales_tx
gc.collect()

describe_df('df', df)
df.head()

df         rows:  8,367,651   cols:     5   memory:     67.1 MB


,item_id,dept_id,store_id,d,sales
0,FOODS_1_001,FOODS_1,TX_1,1,0
1,FOODS_1_002,FOODS_1,TX_1,1,0
2,FOODS_1_003,FOODS_1,TX_1,1,0
3,FOODS_1_004,FOODS_1,TX_1,1,0
4,FOODS_1_005,FOODS_1,TX_1,1,2


In [13]:
# Every series should have every day exactly once
assert len(df) == 4311 * 1941

# No units lost or duplicated in the reshape
assert df['sales'].sum() == total_units_before

# No missing values
assert df.isna().sum().sum() == 0

# Days run from 1 to 1941 with nothing missing
assert df['d'].min() == 1 and df['d'].max() == 1941

print("✅ Reshape verified:", f"{len(df):,} rows, {df['sales'].sum():,} total units")

✅ Reshape verified: 8,367,651 rows, 13,172,106 total units


### Step 3 — Result

**8,367,651 rows × 5 columns in 67 MB** — less memory than the 4,311-row wide table used before preparation (68.5 MB). Total units sold (13,172,106) is identical before and after the reshape, confirming no data was lost or duplicated.

---

## Step 4 — Merge Calendar (Dates, Events, SNAP)

We attach calendar information to every row with a **left, many-to-one merge** on the day number `d`. Before merging, the calendar is trimmed and converted to compact types, since every merged column is copied onto 8.4M rows.

- Keep: `date`, `wm_yr_wk` (link to prices), event columns, `snap_TX` (renamed `snap`)
- Drop: other states' SNAP flags, and date parts (`weekday`, `month`, …) that can be derived from `date` later
- `validate='many_to_one'` guards against duplicate calendar keys silently duplicating sales rows

In [14]:
cal = calendar[['d', 'date', 'wm_yr_wk',
                'event_name_1', 'event_type_1',
                'event_name_2', 'event_type_2',
                f'snap_{STATE}']].copy()

# 'd_1' -> 1, matching the type of df['d']
cal['d'] = cal['d'].str[2:].astype('int16')

# Text -> real datetime
cal['date'] = pd.to_datetime(cal['date'])

# Compact types
cal['wm_yr_wk'] = cal['wm_yr_wk'].astype('int16')
for col in ['event_name_1', 'event_type_1', 'event_name_2', 'event_type_2']:
    cal[col] = cal[col].astype('category')

cal = cal.rename(columns={f'snap_{STATE}': 'snap'})
cal['snap'] = cal['snap'].astype('int8')

# The key must be unique: one row per day
assert cal['d'].is_unique

print(cal.dtypes)
cal.head()

d                        int16
date            datetime64[ns]
wm_yr_wk                 int16
event_name_1          category
event_type_1          category
event_name_2          category
event_type_2          category
snap                      int8
dtype: object


,d,date,wm_yr_wk,event_name_1,event_type_1,event_name_2,event_type_2,snap
0,1,2011-01-29,11101,NaN,NaN,NaN,NaN,0
1,2,2011-01-30,11101,NaN,NaN,NaN,NaN,0
2,3,2011-01-31,11101,NaN,NaN,NaN,NaN,0
3,4,2011-02-01,11101,NaN,NaN,NaN,NaN,1
4,5,2011-02-02,11101,NaN,NaN,NaN,NaN,0


In [15]:
rows_before = len(df)

df = df.merge(cal, on='d', how='left', validate='many_to_one')

# A left many-to-one merge must NOT change the number of rows
assert len(df) == rows_before

# Every sales day must have found its calendar row
assert df['date'].isna().sum() == 0

print("Date range:", df['date'].min().date(), "→", df['date'].max().date())
describe_df('df', df)
df.head()

Date range: 2011-01-29 → 2016-05-22
df         rows:  8,367,651   cols:    12   memory:    192.6 MB


,item_id,dept_id,store_id,d,sales,date,wm_yr_wk,event_name_1,event_type_1,event_name_2,event_type_2,snap
0,FOODS_1_001,FOODS_1,TX_1,1,0,2011-01-29,11101,NaN,NaN,NaN,NaN,0
1,FOODS_1_002,FOODS_1,TX_1,1,0,2011-01-29,11101,NaN,NaN,NaN,NaN,0
2,FOODS_1_003,FOODS_1,TX_1,1,0,2011-01-29,11101,NaN,NaN,NaN,NaN,0
3,FOODS_1_004,FOODS_1,TX_1,1,0,2011-01-29,11101,NaN,NaN,NaN,NaN,0
4,FOODS_1_005,FOODS_1,TX_1,1,2,2011-01-29,11101,NaN,NaN,NaN,NaN,0


In [16]:
# How many days have each event type?
print("Days per event type:")
print(cal.loc[cal['d'] <= 1941, 'event_type_1'].value_counts())

# Share of days that are SNAP days in Texas
snap_share = cal.loc[cal['d'] <= 1941, 'snap'].mean()
print(f"\nShare of days that are SNAP days in {STATE}: {snap_share:.1%}")

# A few example rows on event days
df[df['event_name_1'].notna()].drop_duplicates('date').head()

Days per event type:
event_type_1
Religious    54
National     51
Cultural     37
Sporting     16
Name: count, dtype: int64

Share of days that are SNAP days in TX: 33.0%


,item_id,dept_id,store_id,d,sales,date,wm_yr_wk,event_name_1,event_type_1,event_name_2,event_type_2,snap
34488,FOODS_1_001,FOODS_1,TX_1,9,2,2011-02-06,11102,SuperBowl,Sporting,NaN,NaN,1
68976,FOODS_1_001,FOODS_1,TX_1,17,0,2011-02-14,11103,ValentinesDay,Cultural,NaN,NaN,0
99153,FOODS_1_001,FOODS_1,TX_1,24,2,2011-02-21,11104,PresidentsDay,National,NaN,NaN,0
168129,FOODS_1_001,FOODS_1,TX_1,40,0,2011-03-09,11106,LentStart,Religious,NaN,NaN,1
198306,FOODS_1_001,FOODS_1,TX_1,47,1,2011-03-16,11107,LentWeek2,Religious,NaN,NaN,0


### Step 4 — Result

Calendar merged with no change in row count; dates span **2011-01-29 → 2016-05-22**. Memory: **192.6 MB**.
- **Events are rare:** 158 of 1,941 days (~8%) have an event (Religious 54, National 51, Cultural 37, Sporting 16).
- **SNAP days:** 33.0% of days in Texas (10 days per month).

---

## Step 5 — Merge Sell Prices

Prices vary by **store, item, and week**, so we merge on all three keys (`store_id`, `item_id`, `wm_yr_wk`). All 7 days in a Walmart week receive that week's price.

The price key columns are first converted to `category` with **the same categories as `df`** — otherwise pandas would fall back to text and inflate memory.

After merging, a missing `sell_price` means the item was **not yet on the shelf**. We verify this (no sales on those days, and missing prices only occur before an item's first price), then remove those rows so the model doesn't mistake "not stocked" for "zero demand".

In [17]:
# Match df's category lists exactly, so the merge keeps the compact type
for col in ['store_id', 'item_id']:
    prices_tx[col] = pd.Categorical(prices_tx[col], categories=df[col].cat.categories)

prices_tx['wm_yr_wk'] = prices_tx['wm_yr_wk'].astype('int16')
prices_tx['sell_price'] = prices_tx['sell_price'].astype('float32')

# Every price row must map to a known store/item (no NaN created by the conversion)
assert prices_tx[['store_id', 'item_id']].isna().sum().sum() == 0

# Exactly one price per store-item-week (otherwise the merge would duplicate rows)
assert not prices_tx.duplicated(subset=['store_id', 'item_id', 'wm_yr_wk']).any()

describe_df('prices_tx', prices_tx)

prices_tx  rows:    987,960   cols:     4   memory:      9.0 MB


In [18]:
rows_before = len(df)

df = df.merge(
    prices_tx,
    on=['store_id', 'item_id', 'wm_yr_wk'],
    how='left',
    validate='many_to_one'
)

assert len(df) == rows_before

del prices_tx
gc.collect()

print(df.dtypes)
describe_df('df', df)

item_id               category
dept_id               category
store_id              category
d                        int16
sales                    int16
date            datetime64[ns]
wm_yr_wk                 int16
event_name_1          category
event_type_1          category
event_name_2          category
event_type_2          category
snap                      int8
sell_price             float32
dtype: object
df         rows:  8,367,651   cols:    13   memory:    226.1 MB


In [19]:
no_price = df['sell_price'].isna()

print(f"Rows with no price: {no_price.sum():,} ({no_price.mean():.1%} of all rows)")
print(f"Units sold on no-price days: {df.loc[no_price, 'sales'].sum():,}")

# Has this series had a price at any point up to this day?
# (1 once the item is first priced, and it stays 1 from then on)
ever_priced = (
    df['sell_price'].notna().astype('int8')
      .groupby([df['item_id'], df['store_id']], observed=True)
      .cummax()
)

# A "gap" = no price on a day AFTER the item was already on sale
gaps = (no_price & (ever_priced == 1)).sum()
print(f"Missing prices after an item's first price (gaps): {gaps:,}")

Rows with no price: 1,594,194 (19.1% of all rows)
Units sold on no-price days: 0
Missing prices after an item's first price (gaps): 0


In [20]:
total_units_before = df['sales'].sum()
rows_before = len(df)

df = df[df['sell_price'].notna()].reset_index(drop=True)

rows_removed = rows_before - len(df)
print(f"Removed {rows_removed:,} pre-launch rows ({rows_removed / rows_before:.1%})")

# Removing these rows must not remove any actual sales
assert df['sales'].sum() == total_units_before
assert df['sell_price'].isna().sum() == 0

describe_df('df', df)
print(f"\n✅ {len(df):,} rows, {df['sales'].sum():,} total units, no missing prices")

Removed 1,594,194 pre-launch rows (19.1%)
df         rows:  6,773,457   cols:    13   memory:    183.0 MB

✅ 6,773,457 rows, 13,172,106 total units, no missing prices


### Step 5 — Result

Prices merged on `store_id` + `item_id` + `wm_yr_wk` with no change in row count and ID columns still `category` (226.1 MB).

**Pre-launch rows verified and removed:**
- 1,594,194 rows (19.1%) had no price
- **0 units** were sold on those days
- **0 gaps**: missing prices occur *only* before an item's first price, never mid-history

So missing price = item not yet on the shelf. These rows were removed so the model doesn't learn "not stocked" as "zero demand". Total units sold remains **13,172,106** (unchanged).

---

## Step 6 — Sort & Save

Rows are sorted by `store_id → item_id → date` so each series is contiguous and in time order (needed for lag and rolling features later). The result is saved as **Parquet** to Google Drive — compressed, fast to load, and it preserves our compact data types (CSV would not).

In [21]:
df = df.sort_values(['store_id', 'item_id', 'date']).reset_index(drop=True)

# Check: within every series, dates only go forward
assert (
    df.groupby(['store_id', 'item_id'], observed=True)['date']
      .apply(lambda s: s.is_monotonic_increasing)
      .all()
)

print("✅ Sorted: each series is contiguous and in date order")
df.head()

✅ Sorted: each series is contiguous and in date order


,item_id,dept_id,store_id,d,sales,date,wm_yr_wk,event_name_1,event_type_1,event_name_2,event_type_2,snap,sell_price
0,FOODS_1_001,FOODS_1,TX_1,1,0,2011-01-29,11101,NaN,NaN,NaN,NaN,0,2.0
1,FOODS_1_001,FOODS_1,TX_1,2,1,2011-01-30,11101,NaN,NaN,NaN,NaN,0,2.0
2,FOODS_1_001,FOODS_1,TX_1,3,1,2011-01-31,11101,NaN,NaN,NaN,NaN,0,2.0
3,FOODS_1_001,FOODS_1,TX_1,4,0,2011-02-01,11101,NaN,NaN,NaN,NaN,1,2.0
4,FOODS_1_001,FOODS_1,TX_1,5,0,2011-02-02,11101,NaN,NaN,NaN,NaN,0,2.0


In [22]:
from google.colab import drive
drive.mount('/content/drive')

# Project folder on your Drive (created if it doesn't exist)
PROJECT_DIR = '/content/drive/MyDrive/m5-inventory'
PROCESSED_DIR = os.path.join(PROJECT_DIR, 'data', 'processed')
os.makedirs(PROCESSED_DIR, exist_ok=True)

OUTPUT_FILE = os.path.join(PROCESSED_DIR, f'm5_{STATE.lower()}_{CATEGORY.lower()}_long.parquet')

df.to_parquet(OUTPUT_FILE, index=False)

size_mb = os.path.getsize(OUTPUT_FILE) / 1e6
print(f"Saved to: {OUTPUT_FILE}")
print(f"File size on disk: {size_mb:.1f} MB")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved to: /content/drive/MyDrive/m5-inventory/data/processed/m5_tx_foods_long.parquet
File size on disk: 25.3 MB


In [23]:
check = pd.read_parquet(OUTPUT_FILE)

# The reloaded table must be identical: same values, same types, same order
pd.testing.assert_frame_equal(df, check)

describe_df('reloaded', check)
print("✅ Saved file matches the in-memory table exactly")

del check
gc.collect()

reloaded   rows:  6,773,457   cols:    13   memory:    183.0 MB
✅ Saved file matches the in-memory table exactly


0

---

## ✅ Summary — Notebook 01

| Stage | Rows | Memory |
|---|---|---|
| Raw files (sales + prices + calendar) | 30,490 series / 6.8M prices | ~1,380 MB |
| Filtered to Texas + FOODS | 4,311 series / 988K prices | ~196 MB |
| Reshaped wide → long | 8,367,651 | 67 MB |
| + calendar | 8,367,651 | 193 MB |
| + prices | 8,367,651 | 226 MB |
| Pre-launch rows removed | **6,773,457** | **183 MB** |

**Key decisions**
- Subset: 3 Texas stores × 1,437 FOODS items = 4,311 series, 2011-01-29 → 2016-05-22
- Compact types (`category`, `int16`, `int8`, `float32`) applied *before* reshaping to keep 8.4M rows in memory
- Every merge is left, many-to-one, validated, and row-count checked
- 1.59M pre-launch rows (no price, zero sales, verified) removed so "not stocked" isn't mistaken for "zero demand"
- Total units sold preserved end-to-end: **13,172,106**

**Output:** `data/processed/m5_tx_foods_long.parquet`

**Next:** Notebook 02 — Exploratory Data Analysis